# Generate QA Dataset for an Existing Corpus

Creates (or recreates) a `train_questions.parquet` for an **already-indexed**
corpus so it can be evaluated with `rag_evaluation.ipynb`.

**Workflow:**
1. Set `NAME` to the target collection (must already have `wiki_corpus.parquet`)
2. Choose QA sources & balancing options
3. Run all cells — loads, enriches, balances, saves
4. Open `rag_evaluation.ipynb` with the same `NAME` and evaluate

In [1]:
from pathlib import Path
import pandas as pd
from config import DATA_DIR, CACHE_DIR

# ── Target collection (must already have wiki_corpus.parquet) ────────────────
NAME = "wiki_full"
COLLECTION_ROOT = Path(DATA_DIR) / NAME
WIKI_PARQUET   = COLLECTION_ROOT / "wiki_corpus.parquet"
QUESTIONS_PATH = COLLECTION_ROOT / "nq_500.parquet"
    
# ── QA sources (HuggingFace config names) ────────────────────────────────────
QA_DATASETS = ['natural_questions']  # Datasets to pull questions from (must be in QUESTIONS_PATH)
POPULARITY_DATASET = "Cyro1/enwiki_pageviews_m"

# ── Balancing ────────────────────────────────────────────────────────────────
BALANCE = True                               # Whether to balance questions across popularity deciles
TARGET_PER_DECILE = 500                      # None → downsample to smallest decile

# ── Synthetic generation (optional) ─────────────────────────────────────────
GENERATE_SYNTHETIC = False
QUESTIONS_PER_DECILE = 300
MODEL_NAME = "gpt-4.1-nano"

# ── Sanity check ─────────────────────────────────────────────────────────────
assert WIKI_PARQUET.exists(), f"Corpus not found: {WIKI_PARQUET}"
print(f"✓ Collection: {NAME}")
print(f"  Corpus:     {WIKI_PARQUET}  ({WIKI_PARQUET.stat().st_size / 1e9:.2f} GB)")
print(f"  QA sources: {QA_DATASETS}")
print(f"  Balance:    {BALANCE}  |  Synthetic: {GENERATE_SYNTHETIC}")

✓ Collection: wiki_full
  Corpus:     /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full/wiki_corpus.parquet  (9.88 GB)
  QA sources: ['natural_questions']
  Balance:    True  |  Synthetic: False


In [2]:
from scripts.prepare_qa_dataset import prepare_qa_dataset

qa_df = prepare_qa_dataset(
    qa_datasets=QA_DATASETS,
    popularity_dataset=POPULARITY_DATASET,
    output_path=QUESTIONS_PATH,
    balance=BALANCE,
    target_per_decile=TARGET_PER_DECILE,
    generate_synthetic=GENERATE_SYNTHETIC,
    corpus_path=WIKI_PARQUET,
    questions_per_decile=QUESTIONS_PER_DECILE,
    model_name=MODEL_NAME,
    cache_dir=CACHE_DIR,
)

INFO - PREPARE QA DATASET
INFO - [1/4] LOAD
/Users/cyro/Documents/VSC/PopularityBias/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
INFO - PyTorch version 2.8.0 available.
INFO - Loading natural_questions...
INFO -   ✓ 81,533 questions
INFO - Total: 81,533 questions
INFO - [2/4] FILTER TO CORPUS
INFO - Loading corpus...
INFO - Kept 81,533 / 81,533 questions (in corpus)
INFO - Corpus has valid deciles (unique: [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
INFO - [3/4] ASSIGN DECILES
INFO - Using deciles from corpus...
INFO - QA decile distribution (from corpus): {0: 28, 1: 10, 2: 25, 3: 43, 4: 90, 5: 148, 6: 383, 7: 1019, 8: 3837, 9: 75950}
INFO - [4/4] BALANCE
INFO - Balancing to 500 per decile...
WARNING - Decile 0: 28 (short by 472)
WARNING - Decile 1: 10 (short by 490)
WARNING - Decile 2: 25 (sh

In [3]:
# ── Check corpus decile distribution ──────────────────────────────────────────
import pyarrow.parquet as pq

print("Checking corpus decile distribution...")
parquet_file = pq.ParquetFile(WIKI_PARQUET)

# Sample deciles from corpus in chunks
decile_counts = {}
sample_size = 0
for i, batch in enumerate(parquet_file.iter_batches(batch_size=100_000, columns=["decile"])):
    batch_df = batch.to_pandas()
    for decile, count in batch_df["decile"].value_counts().items():
        decile_counts[decile] = decile_counts.get(decile, 0) + count
    sample_size += len(batch_df)
    del batch_df
    if i >= 5:  # Check first ~500k docs
        break

print(f"Corpus decile distribution (first {sample_size:,} docs):")
for decile in sorted(decile_counts.keys()):
    print(f"  Decile {decile}: {decile_counts[decile]:,}")
    
if len(decile_counts) == 1 and 0 in decile_counts:
    print("\n⚠️  WARNING: Corpus has ALL documents in decile 0!")
    print("   This is incorrect. The corpus needs proper deciles assigned.")
    print("   You need to regenerate the corpus with correct deciles from popularity data.")

Checking corpus decile distribution...
Corpus decile distribution (first 600,000 docs):
  Decile -1: 1,334
  Decile 0: 60,329
  Decile 1: 58,896
  Decile 2: 58,196
  Decile 3: 60,133
  Decile 4: 59,382
  Decile 5: 60,133
  Decile 6: 59,728
  Decile 7: 59,962
  Decile 8: 60,102
  Decile 9: 61,805


In [4]:
# ── Quick inspection ──────────────────────────────────────────────────────────
print(f"Saved: {QUESTIONS_PATH}")
print(f"Total: {len(qa_df):,} questions\n")

if "decile" in qa_df.columns:
    print("Per-decile distribution:")
    print(qa_df["decile"].value_counts().sort_index())

if "dataset" in qa_df.columns:
    print(f"\nSources:")
    print(qa_df["dataset"].value_counts())

display(qa_df.sample(5, random_state=42))

Saved: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full/nq_500.parquet
Total: 2,227 questions

Per-decile distribution:
decile
0     28
1     10
2     25
3     43
4     90
5    148
6    383
7    500
8    500
9    500
Name: count, dtype: int64

Sources:
dataset
natural_questions    2227
Name: count, dtype: int64


,question_id,question_text,answer_texts,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank,dataset,decile
56,-424707762600672834,how many wizards are there in the lord of the ...,"[five, ]",23141947,Wizard (Middle-earth),23.293561,5.062219e+06,natural_questions,2
494,-845637163433103728,when one of a kitten's eyes was sewn shut the ...,"[ocular dominance columns, ]",27391465,Monocular deprivation,164.395833,2.015208e+06,natural_questions,6
1676,8808731297295758463,who wrote the gospel of the holy twelve,"[Rev. Gideon Jasper Richard Ouseley, ]",30747994,The Gospel of the Holy Twelve,1046.041667,9.221636e+05,natural_questions,8
218,7788386234127027369,how many medals did austria win in the 2011 al...,"[8, ]",52733524,Alpine skiing at the Winter Universiade,76.000000,2.911801e+06,natural_questions,5
744,3234021398296878930,where is the world of wearable art held,"[Nelson , New Zealand, ]",13332944,World of Wearable Art,337.979167,1.438631e+06,natural_questions,7


In [5]:
# ── Verify overlap with corpus ────────────────────────────────────────────────
corpus_ids = set(pd.read_parquet(WIKI_PARQUET, columns=["wikipedia_id"])["wikipedia_id"].astype(int))
qa_ids     = set(qa_df["wikipedia_id"].astype(int))

in_corpus = qa_ids & corpus_ids
missing   = qa_ids - corpus_ids

print(f"QA doc IDs in corpus: {len(in_corpus):,} / {len(qa_ids):,}  ({100 * len(in_corpus) / len(qa_ids):.1f}%)")
if missing:
    print(f"⚠️  {len(missing):,} QA doc IDs NOT in corpus — these questions can never be answered correctly")
else:
    print("✓ All QA documents exist in the corpus")

QA doc IDs in corpus: 2,110 / 2,110  (100.0%)
✓ All QA documents exist in the corpus
